# Part 1: A Real Non-Equilibrium Chemistry Solver — and Why It Needs an Emulator

**Learning objectives.** By the end of this notebook you will be able to:

- Describe what makes a chemical reaction network a *stiff* ODE system
- Run a six-species H/He non-equilibrium solver and interpret its output
- Measure the per-cell computational cost of the solver
- Explain why that cost motivates a neural network surrogate

**Prerequisites:** comfortable Python and numpy; no HPC experience needed yet.
The solver lives in `hhe_chemistry.py` in this folder — pure numpy/scipy, so
nothing to compile or crash.

In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt

import hhe_chemistry as chem

print("Solver module loaded.")

## The physics

We track six species — $\textrm{HI, HII, HeI, HeII, HeIII}$, and electrons — coupled by
collisional ionization and recombination, plus five cooling channels
(line excitation, ionization, recombination, and bremsstrahlung).
The rate fits come from Katz, Weinberg & Hernquist (1996), ApJS 105, 19.

The state of one gas cell is `[x_HII, x_HeII, x_HeIII, T]`: three ionization
fractions and a temperature. Reaction timescales span many orders of magnitude
at once, which is the definition of a *stiff* system — an explicit integrator
would need absurdly small steps, so we use an implicit (BDF) solver.

In [ ]:
# Collisional ionization equilibrium (CIE) cooling curve:
# at each fixed T, relax the chemistry to equilibrium, then record
# the cooling rate Lambda / n_H^2.
n_H = 0.1  # cm^-3
T_vals = np.logspace(4, 8, 40)

Lambda_cie = []
for T in T_vals:
    y_eq = chem.equilibrium_state(T, n_H)
    Lambda_cie.append(chem.cooling_rate(y_eq, n_H) / n_H**2)
Lambda_cie = np.array(Lambda_cie)

fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(T_vals, Lambda_cie, lw=2, color="steelblue")
ax.set_xlabel("Temperature [K]")
ax.set_ylabel(r"$\Lambda / n_H^2$  [erg cm$^3$ s$^{-1}$]")
ax.set_title("CIE cooling curve, primordial H/He gas")
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

## Reading the curve

The sharp peak near 2×10⁴ K is collisional excitation of $\textrm{HI}$ (Lyman-alpha) — the moment hydrogen has both neutral atoms and free electrons, it radiates ferociously. The shoulder near 10⁵ K is the same process for HeII. Above ~10⁶ K the gas is fully ionized and only bremsstrahlung remains, rising
slowly as √T. This shape should match Figure 1-style plots in any galaxy formation text; if yours looks different, check the rate functions first.

In [ ]:
# Non-equilibrium cooling tracks: drop hot gas at three densities
# and watch it cool. Higher density -> faster cooling (rate ~ n^2,
# thermal energy ~ n).
Myr = 3.156e13  # seconds
densities = [1e-3, 1e-2, 1e-1]  # number of atoms per cubic cm. 
                                # Values are pretty small (0.5 n_H is typical for the interstellar medium, with ionized regions being 
                                # even lower and molecular clouds being higher).)

fig, ax = plt.subplots(figsize=(8, 5))
for n_H in densities:
    y = chem.equilibrium_state(1e7, n_H)   # start hot, in equilibrium
    times, temps = [0.0], [y[3]]
    for step in range(60):
        y, _ = chem.integrate_cell(y, n_H, 10 * Myr)
        times.append(times[-1] + 10)
        temps.append(y[3])
    ax.semilogy(times, temps, lw=2, label=f"$n_H = {n_H:g}$ cm$^{{-3}}$")

ax.set_xlabel("Time [Myr]")
ax.set_ylabel("Temperature [K]")
ax.set_title("Cooling tracks from $T = 10^7$ K")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## The cost problem

Each call to `integrate_cell` runs a stiff implicit solver: it factorizes a Jacobian (a matrix of derivatives for a multidimensional function) and iterates Newton steps, taking milliseconds per cell. A modern cosmological simulation has 10⁸–10¹⁰ cells and calls the chemistry solver every timestep — chemistry can consume a large fraction of the total runtime.

Let's measure the cost directly. This number is the baseline our emulator has to beat.

In [ ]:
# Time the solver on random states across the parameter space.
rng = np.random.default_rng(0)
N_TIMING = 100

t0 = time.perf_counter()
for _ in range(N_TIMING):
    T = 10 ** rng.uniform(4, 8)
    n_H = 10 ** rng.uniform(-4, 0)
    y0 = [rng.random(), 0.2, 0.2, T]
    chem.integrate_cell(y0, n_H, 10 * Myr)
elapsed = time.perf_counter() - t0

ms_per_cell = 1e3 * elapsed / N_TIMING
print(f"Average solver cost: {ms_per_cell:.2f} ms per cell")
print(f"Cost for one timestep of a 256^3 simulation: "
      f"{256**3 * ms_per_cell / 1e3 / 3600:.1f} CPU-hours")

### Exercise 1 — where is the solver slowest?

The per-cell cost is not uniform: it depends on how far the cell is from equilibrium and how stiff the local rates are. Modify the timing loop to record the cost of each cell separately, then make a scatter plot of cost in the (log T, log n_H) plane. Where are the expensive cells, and can you explain why physically?

*Hint:* `integrate_cell` returns `(y1, nfev)` — the number of RHS evaluations `nfev` is a cleaner cost proxy than wall time (run time is proportional to the number of evaluations)

For the expensive cell explanation, it doesn't have to be purely computational in motivation. Possibly a physical process as well? 

In [ ]:
rng = np.random.default_rng(1)
N = 400 # number of random points to probe
logT = # probe a uniform distribution in log space from 4 to 8
logn = # prob a uniform distribution in log space from -4 to 0
cost = np.empty(N)

for i in range(N):
    # use the timing loop above as guidance if needed for how to call the solver and record the number of RHS evaluations
    # make sure to save the cost calculation to plot as colorbar for your scatter plot

fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(logT, logn, c=cost, s=18, cmap="viridis") #change variable names as needed to tailor to your code
fig.colorbar(sc, label="RHS evaluations (cost proxy)")
ax.set_xlabel(r"log$_{10}$ T [K]")
ax.set_ylabel(r"log$_{10}$ n$_H$ [cm$^{-3}$]")
ax.set_title("Solver cost across parameter space")
plt.tight_layout()
plt.show()

## The surrogate task

We will train a network to replace `integrate_cell` itself:

    input :  (log T, log n_H, x_HII, x_HeII, x_HeIII, log dt)
    output:  (log T', x_HII', x_HeII', x_HeIII')  — the state after dt

This is a *step operator* surrogate, the same idea used in current research
to accelerate ISM chemistry (Branca & Pallottini 2023, MNRAS 518, 5718;
Branca & Pallottini 2024, A&A 684, A203). Note what it is **not**: it is not
a fit to an equilibrium table. Equilibrium cooling is a smooth function of
three variables — a lookup table already solves that problem in nanoseconds,
and no neural network can beat it. The surrogate earns its keep only when
the thing it replaces is genuinely expensive.

## Generating training data on the cluster

We need ~10⁶ solver calls for training. At ~20 ms each that is roughly six CPU-hours — painful in a notebook, trivial for a cluster. Because every
sample is independent, this is an *embarrassingly parallel* workload: the canonical use case for SLURM **tasks** and **job arrays**. Within a single batch job, many tasks can be submitted simultaneously, splitting up the resources allocated to the job to do the work local to each task. On the otherhand, SLURM job arrays submit a sequence of equivalent jobs which all request the same number of resources. In our case, that would mean that each sample runs in its own job. Choosing between job arrays and tasks depends on what you need to run a given computational task and how resources can be allocated on a cluster. On Perlmutter, nodes can only be split up in fractions of half a node (via the `shared` queue). Since our samples require only a few cores to generate, it is more effective in this case to use *tasks* to parallelize our workflow.

From a login node, in this folder:

    sbatch submit_datagen_tasks.sh         
    squeue --me                          # watch your army of jobs
    python merge_chunks.py               # after they finish

Each task runs `generate_data.py` with a different `--chunk-id`, 
writes its own HDF5 chunk (no communication, no file contention), and the
merge script combines them. **Never run heavy generation inside Jupyter**:
a batch job can be checkpointed, retried, and scaled; a notebook kernel
cannot. For a laptop-scale test, run
`python generate_data.py --n-samples 2000` in a terminal instead.